# 01_bronze_transform

**Purpose:** Load the raw source files into Bronze Delta tables as text, unchanged, with lineage columns
(`_source_file`, `_ingest_time`, `_run_id`). Sources are defined in `config/sources.json`.

**Input:** `Files/config/sources.json`; raw files in `Files/raw/petrinex` and `Files/raw/aer_st37`.

**Output tables:**
- `bronze_petrinex_vol` (append by file: the rows of a reloaded file are replaced in one Delta commit with
  `replaceWhere`, so a failed write keeps the old rows)
- `bronze_st37_production_string`, `bronze_st37_bh` (overwrite)
- `pipeline_run_log` (append, one row per file)

**Run order:** step 1 of 3 in the Data Pipeline `pl_well_production_monthly`:
**01_bronze_transform** -> 02_silver_transform -> 03_gold_transform -> semantic model refresh.

**Parameter:** `ONLY_SOURCE` (default `None`) loads every source; set it to one source name from
`config/sources.json` (for example `"petrinex_vol"`) to reload only that source while debugging.

**Failure:** every file is tried and logged. If any file is `FAILED` or `MISMATCH`, the notebook raises after
logging, so the pipeline stops before Silver.


In [ ]:
# Parameters (the pipeline can override them). None loads every source in config/sources.json.
ONLY_SOURCE = None


In [ ]:
import os, json, uuid, fnmatch, re
from datetime import datetime
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

RUN_ID = str(uuid.uuid4())[:8]
BASE = "/lakehouse/default/Files"

with open(f"{BASE}/config/sources.json") as fh:
    CONFIG = json.load(fh)
SOURCE_NAMES = [s["name"] for s in CONFIG["sources"]]
if ONLY_SOURCE is not None and ONLY_SOURCE not in SOURCE_NAMES:
    raise ValueError(f"ONLY_SOURCE '{ONLY_SOURCE}' is not in config/sources.json: {SOURCE_NAMES}")
print("RUN_ID:", RUN_ID, "| ONLY_SOURCE:", ONLY_SOURCE)

for src in CONFIG["sources"]:
    folder = f"{BASE}/{src['folder']}"
    files = [f for f in os.listdir(folder) if fnmatch.fnmatch(f, src["file_pattern"])]
    print(src["name"], "->", files)


def clean_cols(cols):
    return [re.sub(r"[ ,;{}()\n\t=]", "_", c.strip()) for c in cols]

def read_source_file(src, fname):
    rel = f"{src['folder']}/{fname}"
    if src["format"] == "csv":
        sdf = (spark.read.option("header", True)
               .option("inferSchema", False).csv(f"Files/{rel}"))
    elif src["format"] == "xlsx":
        pdf = pd.read_excel(f"{BASE}/{rel}", dtype=str)
        pdf = pdf.astype(object).where(pdf.notna(), None)
        schema = StructType([StructField(c, StringType(), True) for c in pdf.columns])
        sdf = spark.createDataFrame(pdf, schema=schema)
    else:
        raise ValueError(f"unsupported format: {src['format']}")
    return sdf.toDF(*clean_cols(sdf.columns))

def write_bronze(sdf, src, fname):
    tbl = src["target_table"]
    sdf = (sdf.withColumn("_source_file", F.lit(fname))
              .withColumn("_ingest_time", F.current_timestamp())
              .withColumn("_run_id", F.lit(RUN_ID)))
    writer = sdf.write.format("delta")
    if src["load_mode"] == "overwrite":
        writer.mode("overwrite").option("overwriteSchema", "true").saveAsTable(tbl)
    elif spark.catalog.tableExists(tbl):
        # One Delta commit removes the file's old rows and adds the new ones; if the write fails, the old rows stay
        file_filter = "_source_file = '{}'".format(fname.replace("'", "''"))
        writer.mode("overwrite").option("replaceWhere", file_filter).saveAsTable(tbl)
    else:
        writer.mode("append").saveAsTable(tbl)
    return spark.table(tbl).filter(F.col("_source_file") == fname).count()

LOG_SCHEMA = ("run_id string, layer string, source string, file string, "
              "rows_in long, rows_out long, status string, message string, "
              "started_at timestamp, ended_at timestamp")

def log_run(rows):
    (spark.createDataFrame(rows, LOG_SCHEMA)
          .write.format("delta").mode("append").saveAsTable("pipeline_run_log"))



problems = []   # (source, file, status, message) for every FAILED or MISMATCH file
for src in CONFIG["sources"]:
    if ONLY_SOURCE is not None and src["name"] != ONLY_SOURCE:
        continue
    files = sorted(f for f in os.listdir(f"{BASE}/{src['folder']}")
                   if fnmatch.fnmatch(f, src["file_pattern"]))
    for fname in files:
        t0, rows_in, rows_out, status, msg = datetime.now(), None, None, "SUCCESS", ""
        try:
            sdf = read_source_file(src, fname)
            rows_in = sdf.count()
            rows_out = write_bronze(sdf, src, fname)
            if rows_in != rows_out:
                status, msg = "MISMATCH", "rows_in != rows_out"
        except Exception as e:
            status, msg = "FAILED", str(e)[:500]
        log_run([(RUN_ID, "bronze", src["name"], fname, rows_in, rows_out,
                  status, msg, t0, datetime.now())])
        print(src["name"], fname, rows_in, rows_out, status, msg)
        if status != "SUCCESS":
            problems.append((src["name"], fname, status, msg))



display(spark.sql(f"SELECT * FROM pipeline_run_log WHERE run_id = '{RUN_ID}' ORDER BY started_at"))

# Fail the notebook (and the pipeline) after every file has been tried and logged
if problems:
    raise RuntimeError(f"Bronze run {RUN_ID}: {len(problems)} file(s) not loaded: {problems}")
